## M03A解壓縮

本筆記紀錄將 M03A 原始資料解壓縮並整理的過程。
也檢查過原始檔案數量正確，

設計基本流程如下：

1. 挑一天解壓縮 到 資料夾staging
2. 制定清洗規則，並寫好程式碼。 
3. 清洗一天，並將清洗完的資料放到interim。
4. 以上步驟都沒問題的話，直接處理151天 (2026/1/1-2026/5/31)



### 1. 挑一天解壓縮 到 資料夾staging

在資料夾：script中建立：py檔案：extract_m03a_simple.py
(相關程式碼請直接去看)
在專案資料夾的cmd 中 輸入：python scripts/extract_m03a_simple.py

測試看看是否正常

本例子是使用TDCS_M03A_20260101_000000.csv，該筆資料作為解壓縮以及後續的例子。

## 2. 制定清洗規則

同上步驟，挑選TDCS_M03A_20260101_000000.csv 此檔案作為示範。  

建立清洗規則：(要注意必須有一致性，且放到其他檔案都可以適用)  
自己認為的M03A清洗邏輯：每個欄位必須符合以下條件   
A時間 : "年/月/日 小時:分鐘(為5的倍數)"   這樣的格式   例如: 2026/1/1 08:50   
B門架: "數字*2(國道幾號)+道路編碼(通常是英文+數字  但也有兩個英文的，這是道路屬性)+數字*3(門架里程位置) + 'N or S or E or W' (方向)  
C方向: N or S or E or S  
D車種:  只能屬於集合 { 31,32,41,42,5}   
E數量:  只能是非負整數    

門架部分是重點，可以使用正則表達式寫出判斷，然而符合格式，不一定真的在門架之中！  
因此決定建立一個門架的json檔案，這些資料中的門架也不需在這json之中才可以。  




## 3. 寫好清洗的程式碼：clean_m03a_one_day.py

注意不要在這邊執行。這邊只是筆記用。  
執行後檢查是否有問題，

In [ ]:
import json
from pathlib import Path
import pandas as pd

# 1. 路徑設定 (只需要測試一天的資料即可)
# 先設定好三個變數(這邊指的是檔案)：引用path 套件的功能，不需要讓每次都打這麼長一段路徑名稱。

input_dir = Path("data/staging/M03A/20260101")
json_path = Path("data/metadata/gantries_2026_detailed.json")
output_path = Path("data/interim/M03A/M03A_20260101.csv")

# 2. 讀取門架 JSON

with json_path.open("r", encoding="utf-8") as f:
    gantry_data = json.load(f)

# JSON 的 key 就是所有合法門架代號，直接在這邊做成一個門架set。
gantry_set = set(gantry_data)
print("合法門架數量：", len(gantry_set))

# 3. 讀取一天內所有 CSV

columns = ["TimeStamp","GantryID","Direction","VehicleType","Volume",]
# 「遞迴搜尋指定資料夾及其所有子資料夾內的所有 .csv 檔案，並將搜尋結果依照檔名排序後，存入 files 變數中。」
# rglob 代表 Recursive Glob（遞迴搜尋）。它不只找 input_dir 當前目錄，還會深入進去所有的子資料夾，找出副檔名為 .csv 的檔案。
files = sorted(input_dir.rglob("*.csv"))
print("CSV 檔案數量：", len(files))

dfs = []

for file in files:
    df_part = pd.read_csv(file,header=None,names=columns,)
    dfs.append(df_part)

# 將一天的 288 個 CSV 合併
df = pd.concat(dfs,ignore_index=True,)
print("原始資料筆數：", len(df))

# 4. A：時間
time = pd.to_datetime(df["TimeStamp"],errors="coerce",)

A = (time.notna() & (time.dt.minute % 5 == 0) & (time.dt.second == 0))

# 5. B：門架

# B1：門架代號格式正確
B1 = df["GantryID"].str.fullmatch(r"^\d{2}[A-Z]{1,2}\d{3,4}[NSEW]$",na=False,)

# B2：門架確實存在於 JSON
B2 = df["GantryID"].isin(gantry_set)
B = B1 & B2

# 6. C：方向
C1 = df["Direction"].isin({"N", "S", "E", "W"})

# Direction 必須與 GantryID 最後一碼相同
C2 = (df["Direction"] == df["GantryID"].str[-1])

C = C1 & C2

# 7. D：車種
D = df["VehicleType"].isin({31, 32, 41, 42, 5})


# 8. E：車流量
volume = pd.to_numeric(df["Volume"],errors="coerce",)

E = (volume.notna()& (volume >= 0) & (volume % 1 == 0))

# 9. 合併清洗條件
condition = A & B & C & D & E
clean = df[condition].copy()
error = df[~condition].copy()

# 10. 清洗後型別
clean["TimeStamp"] = pd.to_datetime(clean["TimeStamp"])
clean["Volume"] = pd.to_numeric(clean["Volume"]).astype(int)


# 11. 顯示結果

print()
print("清洗結果")
print("原始資料：", len(df))
print("合法資料：", len(clean))
print("異常資料：", len(error))

print()
print("各條件不合格數量")
print("A 時間錯誤：", (~A).sum())
print("B1 門架格式錯誤：", (~B1).sum())
print("B2 門架不存在：", (~B2).sum())
print("C 方向錯誤：", (~C).sum())
print("D 車種錯誤：", (~D).sum())
print("E 數量錯誤：", (~E).sum())


# 12. 輸出清洗結果

output_path.parent.mkdir(parents=True,exist_ok=True,)

clean.to_csv(output_path,index=False,encoding="utf-8-sig",)

print()
print("清洗完成：", output_path)

## 4. 結果檢查：  

合法門架數量： 341  
CSV 檔案數量： 288  
原始資料筆數： 491040  

清洗結果  
原始資料： 491040  
合法資料： 485280  
異常資料： 5760  

各條件不合格數量  
A 時間錯誤： 0  
B1 門架格式錯誤： 0  
B2 門架不存在： 0  
C 方向錯誤： 5760  
D 車種錯誤： 0  
E 數量錯誤： 0  

有趣的結果，在方向中出問題，而這個5760的數字也很有意思，  
5760 = 288 * 20 = 288 * 5 * 4   
意思是：288個csv檔案，每個檔案之中有20個出問題，而20又剛好是5*4，不就正好是5個車種 搭配4個門架。  

因此在這邊猜測：「每天固定有 4 個門架，其 Direction 欄位和 GantryID 最後一碼並不一致。」  

因此嘗試：先把那四個門架找出來！  
在 C = C1 & C2 後面加入：(只看「方向本身合法，但與門架末碼不一致」)  
```py
direction_mismatch = df[C1 & ~C2][
    ["GantryID", "Direction"]
].drop_duplicates()

print(direction_mismatch)
```
得到：
```
835  03A0015N         W  
840  03A0015S         E  
845  03A0041N         W  
850  03A0041S         E  
```

去原始資料檢查也發現：03A0015N ,03A0015S, 03A0041N,03A0041S 這四個門架，雖然門架代號最後是N or S ，可是方向卻是顯示W , E

### 結論：  
最初假設 Direction 必須等於 GantryID 最後一碼。  
實際測試完整一天後，發現共有 5,760 筆不符合。  
經拆解數量與檢查原始資料，確認異常集中於國道 3 甲的四個門架。其門架代號仍以 N/S 結尾，但實際車行方向為 E/W。  
因此，此條件不是普遍成立的資料規則，而只是大多數南北向國道上的規律。  




## 5. 規則修正

保留原本的清洗程式碼： clean_m03a_one_day.py   
不要刪掉，因為這是學習與觀察的機會。
寫一支新的程式碼：clean_m03a_one_day_final.py  


原規則：Direction == GantryID 最後一碼  

修正後：Direction ∈ {N, S, E, W}  即可。  

門架末碼與 Direction 是否一致，改列為資料觀察，而非硬性刪除條件。  

執行成功，清洗正確。也可以發現原始資料非常乾淨。



## 6. 完成五個月

將上述流程寫成一個程式。

最終將5個月的資料都清洗乾淨。

## 心得與總結：
秉持這樣的態度：  
提出假設 → 用資料驗證 → 遇到反例 → 修正模型  

